In [1]:
# @title Imports

import glob
import math
import random
import os
from icecream import ic

import cv2
import einops
from PIL import Image
import scipy.io as sio
import mediapy as media
import numpy as np
import pandas as pd
import queue
import seaborn as sns
import tensorflow as tf
import tqdm
import jax
import sklearn
import matplotlib.pyplot as plt
import cv2

I0000 00:00:1784801843.939497   28188 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784801843.980968   28188 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784801845.151254   28188 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from models.rvm import *

In [3]:
model_variant = 'S'
model = VideoSiamMAE(
    tokenizer=Tokenizer(
        patch_embedding=PatchEmbedding(patch_size=[1, 16, 16], num_features=384),  # confirmed by checkpoint: encoder dim 384
        posenc=SincosPosEmb(base_token_shape=[16, 16]),
    ),
    encoder=Transformer.from_variant_str(variant_str=model_variant, dtype=jax.numpy.bfloat16),  # checkpoint confirms: 384 dim, 6 heads (kernel [384,6,64]), mlp 1536
    rnn_core=GatedTransformerCore(
        transformer=CrossAttentionTransformer(
            num_layers=4,
            num_heads=8,         # paper Table 5; if this errors, read true count off the error (see note)
            num_feats=384,
            mlp_dim=2048,        # ← THE FIX: checkpoint expects (384, 2048), flat value, not 4×dim
            dtype=jax.numpy.bfloat16,
        ),
        initializer=RandomStateInit(),
        token_dim=384,
        state_layer_norm=nn.LayerNorm(epsilon=0.0001, use_scale=True, use_bias=False),
    ),
    latent_emb_dim=384,
    # Decoder: fixed across all sizes — checkpoint confirms 512 dim, 16 heads (kernel [512,16,32]), mlp 2048
    decoder=CrossAttentionTransformer(
        num_layers=8,
        num_heads=16,
        num_feats=512,
        mlp_dim=2048,
        dtype=jax.numpy.bfloat16,
    ),
    decoder_embedder=nn.Dense(512),   # checkpoint confirms: kernel (384, 512)
    delta_embedder=nn.Dense(512),     # checkpoint confirms: kernel (64, 512) — delta is 64-dim Fourier-embedded upstream
    latent_posenc=SincosPosEmb(),
    detokenizer=Detokenizer(patch_size=(16, 16), num_features=3),  # checkpoint confirms: (512, 768) = 16·16·3
    decoder_emb_dim=512,
    masking_ratio=0.85,
)

def recover_tree(flat_dict):
  tree = {}
  for k, v in flat_dict.items():
    parts = k.split("/")
    node = tree
    for part in parts[:-1]:
      if part not in node:
        node[part] = {}
      node = node[part]
    node[parts[-1]] = v
  return tree

restored_params = recover_tree(np.load("/home/geometric-ssl/src/evals/ckpts/pretrain_rvm_small16_256_204031069.npz", allow_pickle=False))
count = sum([np.prod(v.shape) for v in jax.tree_util.tree_leaves(restored_params)])
print(f'number of params {model_variant} = {count}')

number of params S = 67480448


In [8]:
weights = np.load("/home/geometric-ssl/src/evals/ckpts/pretrain_rvm_small16_256_204031069.npz", allow_pickle=False)
weights

NpzFile '/home/geometric-ssl/src/evals/ckpts/pretrain_rvm_small16_256_204031069.npz' with keys: cls_token, decoder/output_norm/bias, decoder/output_norm/scale, decoder/xa_blocks_0/attention/key/bias, decoder/xa_blocks_0/attention/key/kernel...

In [4]:

def init_ema(params):
  """Copy params to use as the initial EMA (teacher) state."""
  return jax.tree_util.tree_map(lambda x: x, params)

@jax.jit
def ema_update(ema_params, online_params, decay=0.999):
  """Polyak-average online_params into ema_params: ema = decay*ema + (1-decay)*online."""
  return jax.tree_util.tree_map(
      lambda ema, online: decay * ema + (1 - decay) * online,
      ema_params, online_params,
  )

# `encoder` is the ViT (Transformer.from_variant_str) inside VideoSiamMAE.
vit_params = restored_params['encoder']
ema_vit_params = init_ema(vit_params)